# ProductIQ Phase 4 — Step 4.16: Candidate Fusion (RRF)

Fuse BM25 and semantic ranked lists with Reciprocal Rank Fusion (k=60). Native BM25/vector scores are preserved for diagnostics only.

Benchmark artifact: `resources/evaluation/hybrid_rrf_benchmark_v1_run.json` (BM25 vs semantic vs RRF P@K / R@K / MRR).

In [ ]:
from pathlib import Path

from productiq.config.settings import get_settings
from productiq.database.engine import create_engine_from_settings
from productiq.representation.query_contract import build_query_representation
from productiq.retrieval import create_bm25_retriever_from_index_path
from productiq.retrieval.contracts import RetrievalRequest
from productiq.retrieval.embedding_encoder import create_bge_small_en_v15_encoder
from productiq.retrieval.rrf_config import RRFConfig
from productiq.retrieval.rrf_fusion import compute_rrf_score
from productiq.retrieval.rrf_hybrid_retriever import create_rrf_hybrid_retriever
from productiq.retrieval.semantic_retriever import create_semantic_retriever_from_engine

query = build_query_representation("black Nike running shoes")
request = RetrievalRequest(query=query, top_k=10)
config = RRFConfig(rank_constant=60)

settings = get_settings()
engine = create_engine_from_settings(settings)
lexical = create_bm25_retriever_from_index_path(
    Path("resources/processed/bm25_lexical_index.pkl")
)
encoder = create_bge_small_en_v15_encoder(device="cpu", allow_cpu_fallback=True)
semantic = create_semantic_retriever_from_engine(engine, encoder)
rrf = create_rrf_hybrid_retriever(lexical, semantic, config=config)

bm25_response = lexical.retrieve(request)
semantic_response = semantic.retrieve(request)
fused = rrf.retrieve(request)

print("BM25 IDs:", [c.product_id for c in bm25_response.candidates[:5]])
print("Semantic IDs:", [c.product_id for c in semantic_response.candidates[:5]])
print("Fused metadata:", fused.metadata.model_dump() if fused.metadata else None)
for row in fused.candidates[:5]:
    print(
        row.product_id,
        "fusion=", row.fusion_score,
        "bm25_rank=", row.bm25_rank,
        "vector_rank=", row.vector_rank,
        "bm25_score=", row.bm25_score,
        "vector_score=", row.vector_score,
    )

print("Example RRF P1 ranks 1+3:", compute_rrf_score(bm25_rank=1, vector_rank=3, rank_constant=60))
engine.dispose()